# खर्च दावे विश्लेषण

हा नोटबुक स्थानिक पावती चित्रांमधून प्रवास खर्च प्रक्रिया करण्यासाठी प्लगइन्स वापरणारे एजंट तयार करण्याचे कसे करायचे हे दाखवतो, खर्च दावा ईमेल निर्माण करतो, आणि खर्च डेटा पाय चार्ट वापरून दृश्यमान करतो. एजंट कार्य संदर्भानुसार गतिशीलपणे फंक्शन्सची निवड करतात.

पावले:
1. OCR एजंट स्थानिक पावती चित्र प्रक्रिया करतो आणि प्रवास खर्च डेटा काढतो.
2. ईमेल एजंट खर्च दावा ईमेल तयार करतो.

### प्रवास खर्च परिस्थितीचे उदाहरण:
समजा आपण एका व्यावसायिक बैठकीसाठी दुसऱ्या शहरात प्रवास करणारे कर्मचारी आहात. आपल्या कंपनीकडे सर्व वाजवी प्रवास-संबंधित खर्च परत फेडण्याची धोरण आहे. संभाव्य प्रवास खर्चाची तपशीलवार माहिती खालीलप्रमाणे आहे:
- परिवहन:
तुमच्या घरच्या शहरापासून गंतव्य शहरापर्यंत आणि परतीसाठी विमान तिकीट.
एअरपोर्टला आणि एअरपोर्टहून टॅक्सी किंवा राइड-हेलिंग सेवा.
गंतव्य शहरातील स्थानिक परिवहन (सार्वजनिक वाहतूक, भाड्याने कार, किंवा टॅक्सी सारखे).

- निवास:
बैठकीच्या जागेजवळील मध्यम श्रेणीतील व्यावसायिक हॉटेलमध्ये तीन रात्री राहणे.

- जेवण:
कंपनीच्या दैनिक भत्त्याच्या धोरणावर आधारित नाश्ता, दुपारचे जेवण, आणि रात्रीचे जेवण यासाठी दैनिक जेवण भत्ता.

- विविध खर्च:
एअरपोर्टवरील पार्किंग फी.
हॉटेलमधील इंटरनेट प्रवेश शुल्क.
तिकिट किंवा लहान सेवा शुल्क.

- दस्तऐवजीकरण:
तुम्ही सर्व पावत्या (फ्लाइट, टॅक्सी, हॉटेल, जेवण, इ.) आणि खर्च रिपोर्ट भरलेला परतावा सादर करता.


## आवश्यक लायब्ररी आयात करा

नोटबुकसाठी आवश्यक लायब्ररी आणि मॉड्यूल आयात करा.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## खर्च मॉडेल्स परिभाषित करा

 वैयक्तिक खर्चांसाठी Pydantic मॉडेल तयार करा आणि वापरकर्त्याच्या क्वेरीला संरचित खर्च डेटामध्ये रूपांतरित करण्यासाठी ExpenseFormatter वर्ग तयार करा.

 प्रत्येक खर्च हा या स्वरूपात दर्शविला जाईल:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## साधने निश्चित करणे - ईमेल तयार करणे

खर्च दाव्यासाठी ईमेल तयार करण्यासाठी एक साधन फंक्शन तयार करा.
- हे साधन Microsoft Agent Framework मधील `@tool` डेकोरेटर वापरते.
- ते खर्चांची एकूण रक्कम मोजते आणि तपशील ईमेल बॉडीमध्ये स्वरूपित करते.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# पावती प्रतिमांमधून प्रवास खर्च काढण्यासाठी साधन

पावती प्रतिमांमधून प्रवास खर्च काढण्यासाठी एक साधन फंक्शन तयार करा.
- हे साधन Microsoft Agent Framework मधील `@tool` डेकोरेटर वापरते.
- हे पावती प्रतिमा वाचते, त्याला base64 मध्ये एन्कोड करते, आणि एजंटसाठी विश्लेषणासाठी डेटा URI परत करते.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## खर्चांची प्रक्रिया

एजंट्स परिभाषित करा आणि त्यांना `WorkflowBuilder` वापरून अनुक्रमे कार्यप्रवाहात जोडाः  
- OCR एजंट रसीद प्रतिमेतून रचना केलेला खर्च डेटा `load_receipt_image` साधन वापरून काढतो.  
- ईमेल एजंट काढलेला डेटा घेते आणि `generate_expense_email` साधन वापरून व्यावसायिक खर्च दावा ईमेल तयार करतो.  
- `WorkflowBuilder` सह `add_edge` वापरून अनुक्रमे चेन तयार करा: OCR एजंट → ईमेल एजंट.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## मुख्य फंक्शन

अनुक्रमिक कार्यप्रवाह तयार करा आणि तो चालवा जेणेकरून पावतीची प्रतिमा प्रक्रिया होईल आणि खर्च दावे ईमेल तयार होईल.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सत्यापन सूचना**:
हा दस्तऐवज AI भाषांतर सेवेद्वारे [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून भाषांतरित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात ठेवा की स्वयंचलित भाषांतरांमध्ये चुका किंवा अचूकतेतील त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्त्रोत मानला जावा. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतर करण्याचा सल्ला दिला जातो. या भाषांतराच्या वापरामुळे होणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थव्यवस्थांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
